In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from transformers import WavLMModel
import torchaudio
from tqdm import tqdm
import joblib

from modules.dataset_processing import extract_archives, prepare_all_split_dataframes, subsample_by_class
from modules.attacks import FGSMAttack, FGSMAttackConfig
from modules.models import WavLM_SpoofDetector
from modules.audio_processing import WaveFormExtract

# Add your project root
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / 'ASVspoof5'
MODEL_DIR = PROJECT_ROOT / 'models'
CHECKPOINT_DIR = MODEL_DIR / 'wavlm'
CACHE_DIR = DATA_ROOT / 'wavlm_feature_cache'  

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = False
BONAFIDE_SUBSAMPLE = 15000
SPOOF_SUBSAMPLE = 15000
BATCH_SIZE = 16          
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 8
MIN_DELTA = 1e-4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
RANDOM_SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
print({'device': str(DEVICE), 'data_root': str(DATA_ROOT), 'checkpoint_dir': str(CHECKPOINT_DIR)})

In [ ]:
extract_archives(DATA_ROOT)
split_frames = prepare_all_split_dataframes(DATA_ROOT, require_existing_files=True)

if TEST_MODE:
    split_frames['train'] = subsample_by_class(split_frames['train'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['dev'] = subsample_by_class(split_frames['dev'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['eval'] = subsample_by_class(split_frames['eval'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)

for split_name, frame in split_frames.items():
    print(split_name, frame.shape, frame['LABEL'].value_counts().to_dict())

In [ ]:

# Create datasets
train_dataset = WaveFormExtract(split_frames['train'], str(DATA_ROOT))
dev_dataset   = WaveFormExtract(split_frames['dev'],   str(DATA_ROOT))
eval_dataset  = WaveFormExtract(split_frames['eval'],  str(DATA_ROOT))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
eval_loader  = DataLoader(eval_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("Datasets and loaders ready. No feature scaling needed for WavLM.")

In [ ]:
model = WavLM_SpoofDetector(num_classes=2, freeze_wavlm=True).to(DEVICE)  # Start frozen, unfreeze later if needed

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

class_weights = torch.tensor([5.0, 1.0]).to(DEVICE) 
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)


In [ ]:
# Training loop
best_dev_loss = float('inf')
best_checkpoint_path = CHECKPOINT_DIR / 'best.pt'
latest_checkpoint_path = CHECKPOINT_DIR / 'latest.pt'
epochs_without_improvement = 0

def run_epoch(model, dataloader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_examples = 0

    for waveforms, labels in tqdm(dataloader, desc="Training" if training else "Validating", leave=False):
        waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(waveforms)
        loss = criterion(logits, labels)

        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_examples += labels.shape[0]
        total_loss += loss.item() * labels.shape[0]

    return total_loss / max(total_examples, 1)


for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    dev_loss = run_epoch(model, dev_loader, optimizer=None)
    scheduler.step(dev_loss)

    # Save checkpoint logic
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'dev_loss': dev_loss,
    }
    torch.save(checkpoint, CHECKPOINT_DIR / f'epoch_{epoch:02d}.pt')
    torch.save(checkpoint, latest_checkpoint_path)

    improved = dev_loss < (best_dev_loss - MIN_DELTA)
    if improved:
        best_dev_loss = dev_loss
        epochs_without_improvement = 0
        torch.save(checkpoint, best_checkpoint_path)
    else:
        epochs_without_improvement += 1

    print({
        'epoch': epoch,
        'train_loss': round(train_loss, 5),
        'dev_loss': round(dev_loss, 5),
        'lr': optimizer.param_groups[0]['lr'],
        'best_saved': improved,
        'epochs_without_improvement': epochs_without_improvement,
    })

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch} (best_dev_loss={best_dev_loss:.5f})")
        break

In [ ]:
best_checkpoint = torch.load(best_checkpoint_path, weights_only=True, map_location=DEVICE)
model.load_state_dict(best_checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {best_checkpoint['epoch']}")

In [ ]:
# FGSM setup 
fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=0.01,          # Start small for waveforms; tune based on results
        clamp_min=-1.0,
        clamp_max=1.0,
        targeted=False,
    )
)


In [ ]:

# Evaluation loop
model.eval()
all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

for waveforms, labels in tqdm(eval_loader, desc="Evaluating"):
    waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)

    # Clean
    with torch.no_grad():
        clean_logits = model(waveforms)
        clean_prob = torch.softmax(clean_logits, dim=1)
        clean_pred = clean_prob.argmax(dim=1)
        clean_true_prob = clean_prob.gather(1, labels.unsqueeze(1)).squeeze(1)

    # Adversarial
    attack_result = fgsm_attack.generate(model=model, inputs=waveforms, labels=labels)
    adv_waveforms = attack_result.adversarial_inputs

    with torch.no_grad():
        adv_logits = model(adv_waveforms)
        adv_prob = torch.softmax(adv_logits, dim=1)
        adv_pred = adv_prob.argmax(dim=1)
        adv_true_prob = adv_prob.gather(1, labels.unsqueeze(1)).squeeze(1)

    all_labels.append(labels.cpu())
    all_clean_predictions.append(clean_pred.cpu())
    all_adversarial_predictions.append(adv_pred.cpu())
    all_clean_true_probabilities.append(clean_true_prob.cpu())
    all_adversarial_true_probabilities.append(adv_true_prob.cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)



In [ ]:
# Metrics 
clean_accuracy = (all_clean_predictions == all_labels).float().mean().item()
adversarial_accuracy = (all_adversarial_predictions == all_labels).float().mean().item()
delta_prob = all_clean_true_probabilities - all_adversarial_true_probabilities
mscd = (delta_prob ** 2).mean().item()
asr = (all_clean_predictions != all_adversarial_predictions).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy:.4f}")
print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")
print(f"MSCD: {mscd:.6f}")
print(f"ASR: {asr:.4f}")